In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("lab09.ipynb")

# Lab 09: Text Processing and Regular Expressions
**DATA2201 — Foundations of Data Science | Fall 2026 | 100 points**

Real-world data often contain inconsistent text and numerical values stored as strings. In this lab, you will use text processing and regular expressions to clean text, extract identifiers and measurements, and organize information into structured tables. These skills support data preparation for statistical analysis and regression modeling.

The exercises apply Lec16 concepts through new examples. All example data are supplied in the notebook; no external dataset is needed.

## Learning objectives
- Standardize inconsistent text labels using regex substitutions.
- Match complete identifiers and extract repeated values from text.
- Use capturing groups to turn text into structured fields.
- Use quantifiers, alternation, word boundaries, and backreferences.
- Clean and de-identify text while preserving useful information.

## Instructions and grading
- **10 questions, 10 points each; 100% autograded.** Each question has public and hidden tests.
- Complete the named pattern/replacement variables. No function definitions are required. Keep the supplied processing code unchanged.
- Use the examples to understand the format, then write a general pattern. Hidden tests follow the same stated requirements.
- Patterns use ASCII letters and digits unless stated otherwise. Use raw strings for regex patterns and replacement backreferences.
- No plots or written responses are required. Work from top to bottom, then restart the kernel and run all cells before submitting.

---

<a id="outline"></a>

## Question Navigation

- [Q1. Canonicalize text labels](#q1)
- [Q2. Validate student IDs](#q2) 
- [Q3. Extract course codes](#q3)
- [Q4. Extract email usernames](#q4)
- [Q5. Extract study hours](#q5)
- [Q6. Split skills fields](#q6)
- [Q7. Remove repeated words](#q7)
- [Q8. Match text patterns](#q8)
- [Q9. Redact student IDs](#q9)

## Setup
Use `re.fullmatch` when the whole value must match, `re.findall` for repeated extraction, `re.sub` for cleaning, and `re.split` for splitting text. In pandas, `.str.extract` returns captured fields and `.str.findall` returns a list of matches for each row.

In [ ]:
import re
import pandas as pd

# Small, original teaching examples; all required data are included below.
print("Ready to clean and extract text.")

<a id="q1"></a>

### Question 1: Canonicalization with regex (12 points)

[Return to the Question Navigation](#outline)

Complete the regex variables below. Lowercase the text, remove all whitespace (including tabs and newlines), replace `&` with `and`, remove literal periods, then remove every occurrence of `county` and `parish`. Apply these steps in this order. Other characters are unchanged. The supplied code produces the resulting string.

**Examples:** `' St. Louis County '` → `'stlouis'`; `'Lewis & Clark County'` → `'lewisandclark'`.

**Hint:** Use `re.sub` with `\s`, an escaped period, and alternation.

**Complete:** `q1_space_pattern`, `q1_amp_pattern`, `q1_dot_pattern`, `q1_suffix_pattern`. The supplied code demonstrates the result; tests apply your patterns to additional inputs.

In [ ]:
q1_space_pattern = ...
q1_amp_pattern = ...
q1_dot_pattern = ...
q1_suffix_pattern = ...

# Supplied example: keep this processing code unchanged.
q1_input = ' St. Louis County '
q1_result = re.sub(q1_suffix_pattern, '', re.sub(q1_dot_pattern, '', re.sub(q1_amp_pattern, 'and', re.sub(q1_space_pattern, '', q1_input.lower()))))
print(q1_result)

In [ ]:
grader.check("q1")

<a id="q2"></a>

### Question 2: Validate student record identifiers (10 points)

[Return to the Question Navigation](#outline)

Before combining survey records, check that each student identifier follows the expected format. Complete `q2_id_pattern` to match an entire identifier:
- Uppercase `S` followed by exactly six digits.

**Examples:** `'S104825'` is valid; `'s104825'`, `'S10482'`, and `'S104825x'` are invalid.

Return values are generated by the supplied code. Do not remove whitespace from invalid values.

**Hint:** Use a literal prefix and a fixed repetition count with `fullmatch`.

In [ ]:
q2_id_pattern = ...

# Supplied processing code: do not change.
student_ids = pd.Series(["S104825", "s104825", "S10482", "S104825x", "S000007"])
q2_valid_ids = student_ids.str.fullmatch(q2_id_pattern)
print(q2_valid_ids)

In [ ]:
grader.check("q2")

<a id="q3"></a>

### Question 3: Extract course codes from advising notes (10 points)

[Return to the Question Navigation](#outline)

An advising note can mention several courses. Complete `q3_course_pattern` to extract every course code in occurrence order, including duplicates.

A code contains **2–4 uppercase letters followed immediately by four digits**, with no space. Require word boundaries at both ends so a code is not extracted from inside a longer word or identifier. Do not use capturing groups.

**Example:** `'Take DATA2201 before CS4801.'` → `['DATA2201', 'CS4801']`.

**Hint:** Combine letter/digit character classes, quantifiers, and `\b`.

In [ ]:
q3_course_pattern = ...

# Supplied processing code: do not change.
advising_notes = pd.Series([
    "Take DATA2201 before CS4801.",
    "Compare STAT2000 with DATA2201; review DATA2201 again.",
    "No courses listed."
])
q3_courses = advising_notes.str.findall(q3_course_pattern)
print(q3_courses)

In [ ]:
grader.check("q3")

<a id="q4"></a>

### Question 4: Extract university email usernames (12 points)

[Return to the Question Navigation](#outline)

Survey comments may include university email addresses. Complete `q4_email_pattern` with **one capturing group** to extract only the username of addresses ending in the literal domain `@mtu.edu`.

A username starts with a lowercase letter, followed by zero or more lowercase letters, digits, or underscores. Matching is case-sensitive. In these inputs, candidate addresses are separated by whitespace, commas, or semicolons, and there are no longer domains such as `mtu.edu.org`.

**Example:** `'Contact maya_2@mtu.edu or lee@mtu.edu'` → `['maya_2', 'lee']`.

Reject other domains and malformed usernames. Use word boundaries around the entire address to avoid extracting from inside a longer word.

**Hint:** Capture the username only; escape the domain's period.

In [ ]:
q4_email_pattern = ...

# Supplied processing code: do not change.
contact_notes = pd.Series([
    "Contact maya_2@mtu.edu or lee@mtu.edu",
    "External contact: lee@example.com",
    "Use alex7@mtu.edu; alex7@mtu.edu"
])
q4_usernames = contact_notes.str.findall(q4_email_pattern)
print(q4_usernames)

In [ ]:
grader.check("q4")

<a id="q5"></a>

### Question 5: Extract reported study hours (12 points)

[Return to the Question Navigation](#outline)

Students report weekly study time in short text responses. Complete `q5_hours_pattern` with **one capturing group** for the number.

Match a nonnegative integer or a decimal with digits on both sides of the period, followed by **one literal space** and either `hour` or `hours`. Match whole words. Return the captured numbers as strings; the supplied code converts them to numbers.

Assume responses contain no signed numbers, malformed decimals, or other measurements. At most one study-time value appears per response. Responses without a match produce a missing value.

**Examples:** `'About 7.5 hours weekly'` → `'7.5'`; `'1 hour'` → `'1'`.

**Hint:** Capture digits followed by an optional decimal part. Use a noncapturing group for that optional part.

In [ ]:
q5_hours_pattern = ...

# Supplied processing code: do not change.
study_responses = pd.Series(["About 7.5 hours weekly", "1 hour", "Usually 12 hours", "Not reported"])
q5_hours_text = study_responses.str.extract(q5_hours_pattern, expand=False)
q5_hours = pd.to_numeric(q5_hours_text)
print(q5_hours)

In [ ]:
grader.check("q5")

<a id="q6"></a>

### Question 6: Split a skills field into separate values (12 points)

[Return to the Question Navigation](#outline)

A survey's skills field uses commas, semicolons, or slashes as separators. Complete `q6_split_pattern` to split on **one or more of those separators**, including any whitespace immediately before or after them.

Preserve the case and internal spaces in each skill name. Assume nonempty input with no leading/trailing separators, no outer whitespace, and no separators inside a skill name.

**Example:** `'Python, SQL; Data visualization / Git'` → `['Python', 'SQL', 'Data visualization', 'Git']`.

**Hint:** Put the three delimiter characters in a character class; surround it with optional whitespace.

In [ ]:
q6_split_pattern = ...

# Supplied processing code: do not change.
skills_text = "Python, SQL; Data visualization / Git"
q6_skills = re.split(q6_split_pattern, skills_text)
q6_skills

In [ ]:
grader.check("q6")

<a id="q7"></a>

### Question 7: Remove consecutive repeated words (12 points)

[Return to the Question Navigation](#outline)

Typing errors can create consecutive duplicate words in comments. Complete `q7_repeat_pattern` and `q7_replacement` to collapse any consecutive run of the same word to its first occurrence, ignoring case.

Words contain ASCII letters only. Repetitions are separated by whitespace, not punctuation. Preserve the first word's case and all text outside the replaced run.

**Example:** `'The the THE examples were were helpful.'` → `'The examples were helpful.'`.

**Hint:** Capture a whole word, then match one or more whitespace-plus-backreference repetitions. Replace with the first captured word.

In [ ]:
q7_repeat_pattern = ...
q7_replacement = ...

# Supplied processing code: do not change.
duplicate_comment = "The the THE examples were were helpful."
q7_comment = re.sub(q7_repeat_pattern, q7_replacement, duplicate_comment, flags=re.IGNORECASE)
q7_comment

In [ ]:
grader.check("q7")

<a id="q8"></a>

### Question 8 - Pattern Match (10 points)

[Return to the Question Navigation](#outline)

Create a regular expression pattern that matches all the positive examples below, but none of the negative examples.  You can not simply list the positives strings "or"ed together. 

| Positive | Negative | 
|----------|----------|
| pit      | pt       | 
| spot     | Pot      |
| spate    | peat     | 
| slap two | part     | 
| respite  | SLIP ten |

In [ ]:
cases = ['pit', 'spot', 'spate', 'slap two', 'respite', 'pt', 'Pot', 'peat', 
         'part', 'SLIP ten']
positive, negative = [], []
pat = r'...'      # Write regular expression pattern here 

# DO NOT CHANGE BELOW
print('Positive Cases: \n')
for ex in cases: 
    match = re.search(pat, ex)
    if ex=="pt": 
        print("\nNegative Cases: \n")
    if match: 
        print("%9s: found" % ex)
        positive.append(ex)
    else: 
        print("%9s: not found" % ex)
        negative.append(ex)

In [ ]:
grader.check("q8")

<a id="q9"></a>

### Question 9: Redact student IDs before sharing comments (10 points)

[Return to the Question Navigation](#outline)

Before sharing comments with classmates, replace valid student IDs with `[ID]`. Complete `q9_id_pattern` to find `S` followed by exactly six digits as a whole word. Match uppercase `S` only.

The supplied code replaces every occurrence while preserving other text. Do not replace a valid-looking substring inside a longer identifier.

**Example:** `'S104825 worked with S000007.'` → `'[ID] worked with [ID].'`.

`'XS104825'`, `'S1048257'`, and `'S104825_extra'` must remain unchanged.

**Hint:** Add word boundaries around the identifier pattern.

In [ ]:
q9_id_pattern = ...

# Supplied processing code: do not change.
sharing_comments = ["S104825 worked with S000007.", "Ask S104825, then thank S104825.", "No student IDs here."]
q9_redacted = [re.sub(q9_id_pattern, "[ID]", text) for text in sharing_comments]
print(q9_redacted)

In [ ]:
grader.check("q9")

## Submission checklist
Restart the kernel and run all cells in order. Check the available tests, save the notebook, and submit the completed student version following the course instructions.

**Total: 100 autograded points.**

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False)